# Этап A: синтетические запросы и эмбеддинги корпуса

Перед запуском: подключить датасет `ru-fin-retrieval` через **+ Add Input**,
меню **Settings** → **Accelerator** → **GPU T4 ×2**; там же проверить,
что интернет включён (пункт читается как **Turn off internet**).

Порядок действий целиком — в `kaggle/README.md` проекта.


## 1. Проверка видеокарты

Должна быть видна Tesla T4. Если команда не найдена — ускоритель не включён.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 2. Библиотеки

`bitsandbytes` нужен для четырёхбитного квантования модели-генератора:
в обычной точности она не поместилась бы в память карты.


In [ ]:
!pip install -q -U sentence-transformers bitsandbytes accelerate


## 3. Код на место


In [ ]:
import glob, os, shutil, zipfile

# Kaggle монтирует датасеты по-разному: то прямо в /kaggle/input/<имя>,
# то глубже, внутри /kaggle/input/datasets/<владелец>/<имя>. Поэтому путь
# не задаётся, а источник ищется по содержимому: нужен каталог, где лежит
# пакет rufin или корпус. Глубина ограничена, чтобы не обходить всё подряд.
def find_source(root='/kaggle/input', max_depth=4):
    archives = []
    for cur, dirs, files in os.walk(root):
        if cur.count('/') - root.count('/') >= max_depth:
            dirs.clear()
        if 'rufin' in dirs or any(f.startswith('acts.jsonl') for f in files):
            return cur, None
        archives += [os.path.join(cur, f) for f in files if f.endswith('.zip')]
    return (os.path.dirname(archives[0]), archives[0]) if archives else (None, None)


def show_tree(root='/kaggle/input', max_depth=3):
    for cur, dirs, files in os.walk(root):
        depth = cur.count('/') - root.count('/')
        if depth >= max_depth:
            dirs.clear()
        print('   ' * depth + os.path.basename(cur) + '/')
        for f in sorted(files)[:6]:
            print('   ' * (depth + 1) + f)


src, archive = find_source()
if src is None:
    print('что ноутбук видит на входе:')
    show_tree()
    raise SystemExit('датасет не подключён: панель справа -> + Add Input -> ru-fin-retrieval -> +')

print('источник:', src)
if archive:
    print('лежит архивом, распаковываю:', archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall('/kaggle/working')
else:
    for item in glob.glob(os.path.join(src, '*')):
        dst = os.path.join('/kaggle/working', os.path.basename(item))
        if os.path.isdir(item):
            shutil.copytree(item, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dst)

os.chdir('/kaggle/working')
print('\nв рабочей папке:', sorted(os.listdir('.')))

assert os.path.isdir('rufin'), f'пакета rufin нет; в источнике было: {sorted(os.listdir(src))}'
# Kaggle иногда разжимает архив при создании датасета, имя корпуса плавает
assert glob.glob('acts.jsonl*'), 'корпуса нет'
print('всё на месте:', sorted(glob.glob('acts.jsonl*')))


## 4. Пробный прогон

Двадцать актов и одна лёгкая модель: проверяем, что датасет виден,
код импортируется и модели скачиваются. Пара минут.


In [ ]:
!python run_phase_a.py \
    --limit-acts 20 --skip-queries \
    --headline-models e5-small --ablation-configs \
    --out /kaggle/working/test


## 5. Полный прогон

Около часа. Сначала генерация полутора сотен вопросов, потом эмбеддинги
пяти моделей по базовой нарезке и абляционных нарезок на e5-small.

Посчитанное не пересчитывается: если сессия оборвётся, просто запустите
ячейку заново.


In [ ]:
!python run_phase_a.py


## 6. Что получилось


In [ ]:
import json
r = json.load(open('/kaggle/working/report_phase_a.json'))
q = r.get('queries')
if q:
    print(f"запросов принято {q['accepted']}, отклонено {q['rejected']}")
    print('причины отказа:', q['reasons'])
for e in r.get('embeddings', []):
    if 'error' in e:
        print('СБОЙ', e['config'], e['model'], e['error'][:120])
    else:
        print(f"{e['chunks_config']:<12} {e['model']:<12} {e['chunks']:>7} фрагм. "
              f"dim={e['dim']:<5} {e['per_second']:>7} фрагм./с {e['size_mb']:>8} МБ")


## 7. Сохранить и скачать

**Save Version** → **Save & Run All (Commit)**. Без этого результат
пропадёт при закрытии сессии.

Скачать нужно только `queries/synthetic.jsonl` — он маленький. Матрицы
эмбеддингов остаются здесь, их прочитает этап B.
